# Big Data Analytics - Topic Drift Assignment

Thomas Timmermans - 2365790

Milan Muroni - 2262576

**Gekozen Dataset:** [Data Mining / Database Systems]



## 0. Imports & Setup

In [1]:
%pip install bertopic

  Obtaining dependency information for bertopic from https://files.pythonhosted.org/packages/81/d2/2eb2fec18b3d2cf487d5d1d9b484c6483c7f0f1a3c6f64e2bb9cb7972a3f/bertopic-0.17.4-py3-none-any.whl.metadata
  Obtaining dependency information for hdbscan>=0.8.29 from https://files.pythonhosted.org/packages/13/8d/eec0040bd273c9e43409672c8cec1956775aca3ebafad8a9e3f5d25cfd80/hdbscan-0.8.44-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for umap-learn>=0.5.0 from https://files.pythonhosted.org/packages/1b/98/f63318ccbe75c810011fe9233884c5d348d94d90005de1b79e5f93bef9c0/umap_learn-0.5.12-py3-none-any.whl.metadata
  Obtaining dependency information for sentence-transformers>=0.4.1 from https://files.pythonhosted.org/packages/66/ae/aac4fcaf7aa27daee96aa15d74770b38bb5ff50b1d31ccf12becd68c6e8b/sentence_transformers-6.2.0-py3-none-any.whl.metadata
  Obtaining dependency information for scikit-learn>=1.0 from https://files.pythonhosted.org/packages/29/8a/8d1e888fe3d0dd4e2a1f3356da

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
conda-repo-cl

In [1]:
%pip install "huggingface_hub<0.25.0"

  Obtaining dependency information for huggingface_hub<0.25.0 from https://files.pythonhosted.org/packages/57/28/a0b0dd3cca63908045edc300360d6cd8758d4d86eee3fd2b08f00c5a41c4/huggingface_hub-0.24.7-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/417.5 kB ? eta -:--:--
   -- ------------------------------------- 30.7/417.5 kB 1.3 MB/s eta 0:00:01
   ----------------- ---------------------- 184.3/417.5 kB 2.2 MB/s eta 0:00:01
   ---------------------------------------  409.6/417.5 kB 3.6 MB/s eta 0:00:01
   ---------------------------------------- 417.5/417.5 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 2.4.0
    Uninstalling huggingface_hub-2.4.0:
      Successfully uninstalled huggingface_hub-2.4.0
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 6.2.0 requires huggingface-hub<3.0.0,>=1.3.0, but you have huggingface-hub 0.24.7 which is incompatible.
transformers 5.22.1 requires huggingface-hub<3.0,>=1.31.0, but you have huggingface-hub 0.24.7 which is incompatible.


In [2]:
import numpy as np
import pandas as pd
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from bertopic import BERTopic


ImportError: cannot import name 'HfFolder' from 'huggingface_hub' (c:\Users\thoma\anaconda3\Lib\site-packages\huggingface_hub\__init__.py)

---  
## 1. Data Inladen & Preprocessing
Specificeer de gekozen dataset (Data Mining of Database Systems) en bereid de titels voor.

In [32]:
titles = []
with open('data_mining_publications.txt', 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            # Zet de regel (string) om naar een echte Python dictionary
            record = ast.literal_eval(line)
            # Voeg uitsluitend de titel toe
            titles.append(record['title'])
df = pd.DataFrame({"title": titles})

---  
## 2. Part 1: Topic Modeling

### 2.1 Conventionele Benadering (TF-IDF + Clustering)

#### A TF-IDF met ondersteuning voor n-grams (bijv. "big data")

In [33]:

tfidf_vectorizer = TfidfVectorizer(
    stop_words="english", # verwijder Engelse stopwoorden
    ngram_range=(1, 3), # kijk ook naar combinaties van 2 en 3 woorden
    
)
tfidf_matrix = tfidf_vectorizer.fit_transform(df["title"]) # zet de titels om in een TF-IDF matrix 

#### B Clustering met K-Means

In [34]:
num_clusters = 10  # Experimenteer met verschillende waarden!
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
df["kmeans_cluster"] = kmeans.fit_predict(tfidf_matrix)

c:\Users\thoma\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


#### C Topicbeschrijving per cluster afleiden

In [35]:
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

print("=== TF-IDF + K-Means Topics ===")
for i in range(num_clusters):
    top_words = feature_names[order_centroids[i, :10]]
    print(f"Cluster {i}: {', '.join(top_words)}")

=== TF-IDF + K-Means Topics ===
Cluster 0: data, classification, using, models, discovery, graph, model, prediction, network, analysis
Cluster 1: learning, machine learning, machine, deep, multi, reinforcement, reinforcement learning, label, active, deep learning
Cluster 2: networks, neural, neural networks, graph neural, graph, graph neural networks, social networks, social, neural network, network
Cluster 3: feature, feature selection, selection, unsupervised feature, unsupervised, unsupervised feature selection, multi, data, supervised feature selection, supervised feature
Cluster 4: detection, anomaly, anomaly detection, outlier detection, outlier, community detection, community, based, fraud, fraud detection
Cluster 5: clustering, based, data, based clustering, spectral, spectral clustering, subspace, subspace clustering, hierarchical, using
Cluster 6: based, learning, representation, representation learning, graph, model, graph based, classification, prediction, approach
Cluster 

### 2.2 Text Embedding Benadering (BERTopic)

In [ ]:
topic_model = BERTopic(language="english", calculate_probabilities=False)
topics, probs = topic_model.fit_transform(df["title"])

df["bertopic_cluster"] = topics

In [ ]:
topic_info = topic_model.get_topic_info() # hij maakt misschien 50 "clusters" = topics
print(topic_info.head(10)) # toon de belangrijkste gevonden topics

---  
## 3. Part 2: Topic Evolution (Topic Drift Visualisatie)
Visualiseer hoe het aantal publicaties per onderwerp evolueert door de jaren heen.

---  
## 4. (Optioneel) Extra Challenge: Temporele Clustering
Neem het publicatiejaar mee in de clustering om compacte tijdclusters te bevoordelen.

---  
## 5. Uitvoeringstijd & Geheugengebruik (Voor Rapport)
Gebruik deze cel om runtime en memory op te meten ten behoeve van Sectie 4.2 in het rapport.